# 13 — StatsBomb pilot (T11)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from statsbombpy import sb

sys.path.insert(0, str(Path("..").resolve()))
from src.estimators import ips_with_clipping, snips_estimate
from src.propensity import effective_sample_size
from src.evaluation import bootstrap_estimates, bias_variance_mse

sns.set_theme(style="whitegrid")
np.random.seed(42)

COMPETITION_ID = 11  # La Liga
SEASON_ID      = 27  # 2015/16
N_MATCHES      = 20  # pilotowa próba
RANDOM_STATE   = 42
N_BOOTSTRAP    = 200

FIGURES_DIR = Path("../figures/week11")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("../results/week11")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Wczytanie danych La Liga 2015/16 (20 meczów)


In [ ]:
matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
match_ids = matches['match_id'].tolist()[:N_MATCHES]
print(f"Loading {N_MATCHES} matches...")

all_events = []
for mid in match_ids:
    ev = sb.events(match_id=mid)
    ev['match_id'] = mid
    all_events.append(ev)

events = pd.concat(all_events, ignore_index=True)
passes = events[events['type'] == 'Pass'].copy()
print(f"Total events: {len(events)}, passes: {len(passes)}")


## Feature engineering

Budujemy prosty zestaw cech dla każdego podania.


In [ ]:
def extract_pass_features(passes: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()

    loc = passes['location'].apply(
        lambda x: x if isinstance(x, list) else [np.nan, np.nan]
    )
    df['start_x'] = loc.apply(lambda x: x[0])
    df['start_y'] = loc.apply(lambda x: x[1])

    df['pass_length'] = pd.to_numeric(passes['pass_length'], errors='coerce')
    df['pass_angle']  = pd.to_numeric(passes['pass_angle'],  errors='coerce')

    end_loc = passes['pass_end_location'].apply(
        lambda x: x if isinstance(x, list) else [np.nan, np.nan]
    )
    df['end_x'] = end_loc.apply(lambda x: x[0])
    df['end_y'] = end_loc.apply(lambda x: x[1])

    df['forward_gain'] = df['end_x'] - df['start_x']

    df['under_pressure'] = passes['under_pressure'].fillna(False).astype(int)
    df['minute'] = pd.to_numeric(passes['minute'], errors='coerce')

    df['shot_assist']  = passes['pass_assisted_shot_id'].notna().astype(int)
    df['goal_assist']  = passes['pass_goal_assist'].fillna(False).astype(int)
    df['reward']       = np.maximum(df['shot_assist'], df['goal_assist'])

    df['treatment'] = (df['forward_gain'] > 10).astype(int)

    df = df.dropna(subset=['start_x', 'start_y', 'pass_length', 'pass_angle',
                           'forward_gain', 'minute'])
    return df.reset_index(drop=True)


feat_df = extract_pass_features(passes)
print(f"Passes after feature extraction: {len(feat_df)}")
print(f"Progressive passes (treatment=1): {feat_df['treatment'].mean():.2%}")
print(f"Shot/goal assists (reward=1):     {feat_df['reward'].mean():.4%}")
print(f"\nReward rate:")
print(f"  Progressive pass:  {feat_df[feat_df['treatment']==1]['reward'].mean():.4%}")
print(f"  Non-progressive:   {feat_df[feat_df['treatment']==0]['reward'].mean():.4%}")


## EDA — rozkład progressive passes


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(feat_df['forward_gain'], bins=50, color="steelblue", alpha=0.85)
axes[0].axvline(10, color="red", linestyle="--", label="próg progressive (10m)")
axes[0].set_xlabel("Forward gain (m)")
axes[0].set_ylabel("Count")
axes[0].set_title("Rozkład postępu podań")
axes[0].legend(fontsize=8)

sc = axes[1].scatter(feat_df['start_x'], feat_df['start_y'],
                     c=feat_df['treatment'], cmap='RdYlGn', alpha=0.15, s=3)
axes[1].set_xlim(0, 120)
axes[1].set_ylim(0, 80)
axes[1].set_xlabel("X (m)")
axes[1].set_ylabel("Y (m)")
axes[1].set_title("Pozycja startowa podań\n(zielony = progressive)")
plt.colorbar(sc, ax=axes[1], label="treatment")

feat_df['x_zone'] = pd.cut(feat_df['start_x'], bins=[0, 40, 80, 120],
                            labels=["Obrona", "Środek", "Atak"])
zone_rate = feat_df.groupby('x_zone', observed=True)['treatment'].mean()
axes[2].bar(zone_rate.index, zone_rate.values, color=["steelblue", "darkorange", "seagreen"], alpha=0.85)
axes[2].set_ylabel("% progressive passes")
axes[2].set_title("Progressive pass rate per strefa")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "statsbomb_eda.png", dpi=160)
plt.show()


## OPE (π_eval: 50% progressive)


In [ ]:
feature_cols = ['start_x', 'start_y', 'pass_length', 'pass_angle', 'under_pressure', 'minute']

X = feat_df[feature_cols].values
T = feat_df['treatment'].values
Y = feat_df['reward'].values.astype(np.float64)

X_tr, X_te, T_tr, T_te, Y_tr, Y_te = train_test_split(
    X, T, Y, test_size=0.3, random_state=RANDOM_STATE, stratify=T
)

print(f"Train: {len(X_tr)}, Test: {len(X_te)}")

ps_model = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, C=1.0)
ps_model.fit(X_tr, T_tr)
pscores_te = ps_model.predict_proba(X_te)[:, 1]  # P(T=1|X) dla danych testowych
pscores_te = np.clip(pscores_te, 0.01, 0.99)

print(f"Propensity scores: min={pscores_te.min():.3f}  max={pscores_te.max():.3f}  mean={pscores_te.mean():.3f}")

X_tr_full = np.hstack([X_tr, T_tr.reshape(-1, 1)])
X_te_full = np.hstack([X_te, T_te.reshape(-1, 1)])
X_te_prog = np.hstack([X_te, np.ones((len(X_te), 1))])  # T=1 dla wszystkich

reward_model = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.05,
    subsample=0.8, eval_metric="logloss",
    tree_method="hist", n_jobs=1, random_state=RANDOM_STATE,
)
reward_model.fit(X_tr_full, Y_tr[Y_tr>=0], verbose=False)
er_te = reward_model.predict_proba(X_te_full)[:, 1]   # E[Y|X, T_observed]
er_prog = reward_model.predict_proba(X_te_prog)[:, 1]  # E[Y|X, T=1]
print(f"Reward model — mean ER (observed): {er_te.mean():.4f}")
print(f"Reward model — mean ER (T=1):      {er_prog.mean():.4f}")


## Estymaty OPE


In [ ]:
PI_EVAL = 0.5

ips_weights = np.where(
    T_te == 1,
    PI_EVAL / pscores_te,
    (1 - PI_EVAL) / (1 - pscores_te)
)

v_dm = float(er_prog.mean())

v_ips = float((ips_weights * Y_te).mean())

v_snips = float((ips_weights * Y_te).sum() / ips_weights.sum())

v_dr = float(v_ips + (er_prog - er_te * ips_weights).mean())

v_naive_prog = float(Y_te[T_te == 1].mean()) if (T_te == 1).any() else 0.0
v_naive_all  = float(Y_te.mean())

ess, ess_ratio = effective_sample_size(ips_weights)

print(f"Evaluation policy: {PI_EVAL*100:.0f}% progressive passes\n")
print(f"Naive CTR (all):     {v_naive_all:.4%}")
print(f"Naive CTR (T=1):     {v_naive_prog:.4%}")
print(f"")
print(f"V_DM   = {v_dm:.4%}")
print(f"V_IPS  = {v_ips:.4%}")
print(f"V_SNIPS= {v_snips:.4%}")
print(f"V_DR   = {v_dr:.4%}")
print(f"")
print(f"ESS = {ess:.0f} / {len(ips_weights)} = {ess_ratio:.3f}")


## Bootstrap CI i wizualizacja


In [ ]:
def dm_fn(er_prog, **_):   return float(er_prog.mean())
def ips_fn(Y_te, ips_weights, **_): return float((ips_weights * Y_te).mean())
def snips_fn(Y_te, ips_weights, **_): return float((ips_weights * Y_te).sum() / ips_weights.sum())
def dr_fn(Y_te, ips_weights, er_prog, er_te, **_):
    return float((ips_weights * Y_te).mean() + (er_prog - er_te * ips_weights).mean())

data_boot = dict(Y_te=Y_te, ips_weights=ips_weights, er_prog=er_prog, er_te=er_te)

boots = {
    "DM":    bootstrap_estimates(dm_fn,   N_BOOTSTRAP, RANDOM_STATE, er_prog=er_prog),
    "IPS":   bootstrap_estimates(ips_fn,  N_BOOTSTRAP, RANDOM_STATE, Y_te=Y_te, ips_weights=ips_weights),
    "SNIPS": bootstrap_estimates(snips_fn,N_BOOTSTRAP, RANDOM_STATE, Y_te=Y_te, ips_weights=ips_weights),
    "DR":    bootstrap_estimates(dr_fn,   N_BOOTSTRAP, RANDOM_STATE, **data_boot),
}

rows = []
for name, b in boots.items():
    rows.append({
        "Estimator": name,
        "V̂":         float(np.mean(b)),
        "CI Lower":  float(np.percentile(b, 2.5)),
        "CI Upper":  float(np.percentile(b, 97.5)),
        "CI Width":  float(np.percentile(b, 97.5) - np.percentile(b, 2.5)),
    })

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False, float_format="{:.4%}".format))

result_df.to_csv(RESULTS_DIR / "statsbomb_ope_results.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_e = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

means   = result_df["V̂"].values
ci_lo   = means - result_df["CI Lower"].values
ci_hi   = result_df["CI Upper"].values - means
axes[0].barh(result_df["Estimator"], means,
             xerr=[ci_lo, ci_hi], color=colors_e, alpha=0.85, capsize=6)
axes[0].axvline(v_naive_all, color="gray", linestyle=":", label=f"Naive CTR = {v_naive_all:.3%}")
axes[0].set_xlabel("Policy value V̂")
axes[0].set_title("StatsBomb: OPE — DM vs IPS vs SNIPS vs DR\n(La Liga 2015/16, eval: 50% progressive)")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.3%}"))
axes[0].legend(fontsize=8)

prog_passes = feat_df[feat_df['treatment'] == 1]
sc = axes[1].scatter(
    prog_passes['start_x'], prog_passes['start_y'],
    c=prog_passes['reward'], cmap='RdYlGn', alpha=0.4, s=6,
    vmin=0, vmax=1
)
axes[1].set_xlim(0, 120)
axes[1].set_ylim(0, 80)
axes[1].set_xlabel("X (m)")
axes[1].set_ylabel("Y (m)")
axes[1].set_title("Progressive passes — kolor = reward\n(zielony = shot/goal assist)")
plt.colorbar(sc, ax=axes[1], label="reward")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "statsbomb_ope_results.png", dpi=160)
plt.show()
